# Phase 0 · Step 5 — Companies House API Spike

**Goal:** prove that the Companies House API can answer questions Q1–Q10 (see `docs/02-users-and-questions.md`) _before_ designing anything.

**This notebook answers five things:**

1. Does authentication work?
2. Which endpoint and field answers each question Q1–Q10?
3. How do endpoints behave when there is **no data** (404 vs empty list)?
4. How many API calls does one full report need — and does that fit the rate limit?
5. How fast are the calls (latency)?

**Output:** fill in `docs/data-notes.md` from the findings in section 9.

**Before running:**

1. Register at the Companies House Developer Hub and create an application with a **REST API key** (live environment).
2. Copy `.env.example` to `.env` and set `CH_API_KEY=...`
3. `uv sync` then open this notebook with the project's environment.
   - run `uv init` and `uv add jupyter`
   - Then run `uv sync`
   - start jupyter: `uv run jupyter notebook`
   - **uv =** a fast Python package/project manager that can create and manage virtual environments, install packages, manage Python versions, and lock dependencies.

> ⚠️ Raw responses contain directors' personal data. They are saved to `data/raw/`, which is **git-ignored**. Never commit them. See `docs/legal-and-ethics.md`.


## 1. Setup


In [42]:
import json
import os
import time
from collections import Counter
from pathlib import Path  # makes working with file/folder paths easier.

import httpx  # is a Python library used to communicate with websites/APIs
import pandas as pd
from dotenv import (
    load_dotenv,
)  # allows you to load environment variables from a .env file

load_dotenv(Path("..") / ".env")
API_KEY = os.getenv("CH_API_KEY")
assert API_KEY, "Set CH_API_KEY in .env"

BASE_URL = "https://api.company-information.service.gov.uk"

RAW_DIR = Path("..") / "data" / "raw"
DOCS_DIR = Path("..") / "docs"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option(
    "display.max_colwidth", 80
)  # When displaying a DataFrame, don't make individual columns wider than 80 characters because of long text.

## 2. A small instrumented client

Every call is logged (endpoint, status, latency) so sections 7–8 can measure call counts and speed.
Authentication is HTTP Basic: the API key is the **username**, the password is **empty**.


This code is basically building a small wrapper around the Companies House API that:

1. Creates a place to record every API call.
2. Creates an HTTP client configured for Companies House.
3. Defines a reusable function for making GET requests.
4. Records useful information about each request.
5. Handles rate limiting and bad/non-JSON responses.
6. Defines another function to save API responses as JSON files.


```
Your notebook
     │
     ▼
ch_get("/company/12345678")
     │
     ▼
httpx client
     │
     ▼
Companies House API
     │
     ▼
Response
     │
     ├── status code → 200
     │
     ├── JSON → company data
     │
     └── headers → rate-limit information
     │
     ▼
Add information to CALL_LOG
     │
     ▼
return (200, company_data)
```


In [6]:
CALL_LOG: list[dict] = []  # you can think of CALL_LOG as a history of API calls

client = httpx.Client(base_url=BASE_URL, auth=(API_KEY, ""), timeout=20.0)


def ch_get(
    path: str,
    params: dict | None = None,
    label: str = "",  # This is an optional human-friendly name for the API call
) -> tuple[int, dict | None]:
    """GET a Companies House endpoint. Returns (status_code, json_or_None). Logs every call."""

    start = time.perf_counter()
    resp = client.get(path, params=params)
    elapsed_ms = (time.perf_counter() - start) * 1000
    CALL_LOG.append(
        {
            "label": label or path.split("/")[-1],
            "path": path,
            "status": resp.status_code,  # 200=OK, 404=Not Found, 429=Rate Limited
            "ms": round(elapsed_ms, 1),
            "ratelimit_remain": resp.headers.get("X-Ratelimit-Remain"),
            "ratelimit_limit": resp.headers.get("X-Ratelimit-Limit"),
            "ratelimit_window": resp.headers.get("X-Ratelimit-Window"),
        }
    )
    if resp.status_code == 429:
        print("⚠️ Rate limited (429). Wait and retry later.")
    try:
        body = resp.json() if resp.content else None
    except ValueError:
        body = {"_non_json_body": resp.text[:500]}

    return resp.status_code, body


def save_raw(name: str, data) -> None:
    (RAW_DIR / f"{name}.json").write_text(json.dumps(data, indent=2, default=str))

## 3. Authentication test


In [7]:
status, body = ch_get(
    "/search/companies", {"q": "tesco", "items_per_page": 5}, label="search"
)
print("Status:", status)

print(
    "Rate-limit headers:",
    {k: v for k, v in CALL_LOG[-1].items() if k.startswith("ratelimit")},
)

assert (
    status == 200
), "Auth failed — check the key is a REST key for the LIVE environment"

[(i["title"], i["company_number"], i["company_status"]) for i in body["items"]]

Status: 200
Rate-limit headers: {'ratelimit_remain': '599', 'ratelimit_limit': '600', 'ratelimit_window': '5m'}


[('TESCO PLC', '00445790', 'active'),
 ('KFORD TYRES (GORNAL) LTD', '09384423', 'active'),
 ('TESCO AQUA (FINCO2) LIMITED', '05888957', 'dissolved'),
 ('TESCO AQUA (1LP) LIMITED', 'BR008975', 'closed'),
 ('TESCO ATRATO DEPOT PROPCO LIMITED', '06969533', 'active')]

**Record:** Did auth work first time? What rate-limit headers came back (limit and window)? Do they match the documented 600 requests / 5 minutes?


In [9]:
## Second company search, to show how the rate-limit headers change as you make more calls.
status1, body1 = ch_get(
    "/search/companies", {"q": "ROCK AUTO'S LTD", "items_per_page": 2}, label="search"
)
print("Status:", status1)

print(
    "Rate-limit headers:",
    {k: v for k, v in CALL_LOG[-1].items() if k.startswith("ratelimit")},
)

assert (
    status1 == 200
), "Auth failed — check the key is a REST key for the LIVE environment"

[(i["title"], i["company_number"], i["company_status"]) for i in body1["items"]]

Status: 200
Rate-limit headers: {'ratelimit_remain': '599', 'ratelimit_limit': '600', 'ratelimit_window': '5m'}


[("ROCK AUTO'S LTD", '17313434', 'active'),
 ('ROCK AUTOS LTD', '14102316', 'dissolved')]

## 4. Build a varied test set (~10 companies)

We need different _types_ of company to see how the data looks in each case, especially the edge cases:

| Category                                 | Why                                     |
| ---------------------------------------- | --------------------------------------- |
| Large plc                                | Lots of data, many officers and filings |
| Small new Ltd (< 1 year)                 | Little data                             |
| Dissolved                                | Q1 status handling                      |
| In liquidation / insolvency              | Q7                                      |
| Overdue accounts                         | Q4                                      |
| Has charges                              | Q6                                      |
| No charges                               | "No data" behaviour                     |
| Corporate PSC (owned by another company) | Q5 edge case                            |

Search results include `company_status` and `date_of_creation`, so we filter search results to find candidates.


A **wrapper function** in Python is a function that encapsulates, or "wraps," another function to extend or modify its behavior without changing the original function's source code.


In [10]:
def search(q: str, n: int = 50) -> list[dict]:
    status, body = ch_get(
        "/search/companies", {"q": q, "items_per_page": n}, label="search"
    )
    return body.get("items", []) if status == 200 and body else []


def profile(number: str) -> tuple[int, dict | None]:
    return ch_get(f"/company/{number}", label="profile")


candidates = search("building services", 100) + search("consulting", 100)
print("Candidates:", len(candidates))
print(Counter(c.get("company_status") for c in candidates))

Candidates: 200
Counter({'active': 140, 'dissolved': 55, 'liquidation': 4, 'administration': 1})


In [14]:
candidates[0]

{'address': {'address_line_1': 'Farm Field Court',
  'country': 'England',
  'locality': 'Northampton',
  'postal_code': 'NN3 8AQ',
  'premises': '45'},
 'address_snippet': '45 Farm Field Court, Northampton, England, NN3 8AQ',
 'company_number': '17491796',
 'company_status': 'active',
 'company_type': 'ltd',
 'date_of_creation': '2026-09-30',
 'description': '17491796 - Incorporated on 30 September 2026',
 'description_identifier': ['incorporated-on'],
 'kind': 'searchresults#company',
 'links': {'self': '/company/17491796'},
 'matches': {'snippet': []},
 'snippet': '',
 'title': 'BUILDING SERVICES LTD'}

```
Search for companies
       ↓
Filter them
       ↓
┌─────────────────────────────────┐
│ large_plc    → company number   │
│ dissolved    → company number   │
│ insolvency   → company number   │
│ new_ltd      → company number   │
└─────────────────────────────────┘
       ↓
     TEST_SET
```

`The purpose is:`
Find representative companies now, save their company numbers, and use those same companies consistently throughout the rest of the notebook for testing.


In [11]:
# This code is building a test set of company numbers, where each company represents a particular edge case.

TEST_SET: dict[str, str] = {}  # category -> company_number

# Large plc — add a well-known one you know, e.g. from search
plc = [
    c
    for c in search("tesco plc", 10)
    if c.get("company_type") == "plc" and c.get("company_status") == "active"
]
if plc:
    TEST_SET["large_plc"] = plc[0]["company_number"]

for c in candidates:
    s, created = c.get("company_status"), c.get("date_of_creation", "")
    if s == "dissolved":
        TEST_SET.setdefault("dissolved", c["company_number"])
    if s in ("liquidation", "insolvency-proceedings", "administration", "receivership"):
        TEST_SET.setdefault("insolvency", c["company_number"])
    if s == "active" and created >= "2025-10-01":
        TEST_SET.setdefault("new_ltd", c["company_number"])

TEST_SET

{'large_plc': '00445790',
 'new_ltd': '17491796',
 'dissolved': '12115175',
 'insolvency': '01468547'}

The above are the companies I've chosen to test my API/data-processing code against.


`Note:`
Set "dissolved" to "12345678" only if "dissolved" isn't already in the dictionary.

So if you've already found a dissolved company, another one won't replace it.
That's useful because you only need one representative company per category.


Previously, I used the search results to find things like dissolved, new, and insolvency companies.

`Now:`
"Some of the other categories I need — overdue accounts and charges — aren't available in the search results, so I need to fetch each company's full profile."

In [15]:
# Profiles are needed to find overdue accounts / charges / no charges.
# Limit how many we check so we don't burn the rate limit.
PROFILE_CACHE: dict[str, dict] = {}
active = [c for c in candidates if c.get("company_status") == "active"][:40]

for c in active:
    num = c["company_number"]
    status, p = profile(num)
    if status != 200:
        continue
    PROFILE_CACHE[num] = p
    if p.get("accounts", {}).get("overdue"):
        TEST_SET.setdefault("overdue_accounts", num)
    if p.get("has_charges"):
        TEST_SET.setdefault("has_charges", num)
    else:
        TEST_SET.setdefault("no_charges", num)

TEST_SET

{'large_plc': '00445790',
 'new_ltd': '17491796',
 'dissolved': '12115175',
 'insolvency': '01468547',
 'no_charges': '17491796',
 'overdue_accounts': '12056899',
 'has_charges': '17016019'}

So the above code is essentially doing targeted data collection. I am making a limited number of API calls to find the specific types of companies I need for testing, while caching the profiles so I don't repeatedly hit the API.

STEP 1
```
Search Companies House
        ↓
Find candidates
        ↓
TEST_SET
  ├── large_plc
  ├── dissolved
  ├── insolvency
  └── new_ltd
```

STEP 2
```
Get profiles for up to 40 active companies
        ↓
Inspect profile data
        ↓
TEST_SET
  ├── overdue_accounts
  ├── has_charges
  └── no_charges
```

**Manual step:** if a category is missing, find one by hand on the Companies House website and add it below. Also try to add a company **owned by another company** (corporate PSC).


In [22]:
# TEST_SET["corporate_psc"] = "XXXXXXXX"
# TEST_SET["missing_category"] = "XXXXXXXX"
pd.Series(TEST_SET, name="company_number")

large_plc           00445790
dissolved           12115175
insolvency          01468547
new_ltd             17016019
no_charges          NI648030
overdue_accounts    12056899
has_charges         17016019
Name: company_number, dtype: object

In [16]:
TEST_SET["corporate_psc"] = "03977902"

pd.Series(TEST_SET, name="company_number")

large_plc           00445790
new_ltd             17491796
dissolved           12115175
insolvency          01468547
no_charges          17491796
overdue_accounts    12056899
has_charges         17016019
corporate_psc       03977902
Name: company_number, dtype: str

## 5. Pull every endpoint for every test company

Endpoints under test:

| Endpoint                                        | Questions                        | What you're trying to learn |
| ----------------------------------------------- | -------------------------------- |-------------------------------- |
| `/company/{n}`                                  | Q1, Q2, Q4 (+ flags for Q6, Q7)  |Basic company information, accounts, status, etc. |
| `/company/{n}/officers`                         | Q3, Q9                           |Directors/officers |
| `/company/{n}/filing-history`                   | Q9 (address changes), Q4 context |Filing history and address changes |
| `/company/{n}/persons-with-significant-control` | Q5                               |Who controls the company |
| `/company/{n}/charges`                          | Q6                               |Secured charges |
| `/company/{n}/insolvency`                       | Q7                               |Insolvency information |
| `/officers/{id}/appointments`                   | Q8                               |Other companies an officer is/was appointed to |

We record the **status code for every call** — this is how we learn the "no data" behaviour.


In [19]:
ENDPOINTS = { # This is just a dictionary containing the endpoints I want to test
    "profile": "/company/{n}",
    "officers": "/company/{n}/officers",
    "filing_history": "/company/{n}/filing-history",
    "psc": "/company/{n}/persons-with-significant-control",
    "charges": "/company/{n}/charges",
    "insolvency": "/company/{n}/insolvency",
}

RAW: dict[str, dict[str, dict]] = {} # keeps the actual API responses
status_matrix = [] # keeps a simple summary of HTTP status codes

for category, num in TEST_SET.items():
    RAW[num] = {}
    row = {"category": category, "company_number": num}
    for name, template in ENDPOINTS.items():
        status, body = ch_get(template.format(n=num), label=name)
        RAW[num][name] = {"status": status, "body": body}
        row[name] = status
        save_raw(f"{num}_{name}", {"status": status, "body": body})
    status_matrix.append(row)

pd.DataFrame(status_matrix)

,category,company_number,profile,officers,filing_history,psc,charges,insolvency
0,large_plc,00445790,200,200,200,200,200,404
1,new_ltd,17491796,200,200,200,200,200,404
2,dissolved,12115175,200,200,200,200,200,404
3,insolvency,01468547,200,200,200,200,200,200
4,no_charges,17491796,200,200,200,200,200,404
5,overdue_accounts,12056899,200,200,200,200,200,404
6,has_charges,17016019,200,200,200,200,200,404
7,corporate_psc,03977902,200,200,200,200,200,404


**Record in data-notes:** For each endpoint — what status code comes back when there is **no data**? (e.g. does a company with no charges return `404`, or `200` with an empty list?) A "no data" response is a **valid answer**, not an error. The MCP server must handle this correctly.

Briefly:

- We picked **different types of companies** to test edge cases.
- We called **every relevant Companies House endpoint** for each company.
- We recorded the **HTTP status code and response body**.
- The goal is to learn how the API represents **"no data"**.
- Importantly, **no data is not necessarily an error**.

For example:

```text
Charges endpoint
Company HAS charges    → 200 + charges
Company has NO charges → 200 + empty list
```

Whereas:

```text
Insolvency endpoint
Company has insolvency    → 200 + data
Company has no insolvency → 404
```

So the MCP server needs to understand the **meaning of each endpoint's response**, rather than treating every non-`200` response as an error.

In [20]:
RAW[TEST_SET["has_charges"]]["charges"]

{'status': 200,
 'body': {'etag': 'da72e353b3a070cc9359ba816b4f34dcb5f9f7cf',
  'items': [{'charge_code': '170160190001',
    'charge_number': 1,
    'classification': {'description': 'A registered charge',
     'type': 'charge-description'},
    'created_on': '2026-03-31',
    'delivered_on': '2026-04-13',
    'etag': 'c7255f3b3bebc979184c35d30c394461b9675641',
    'links': {'self': '/company/17016019/charges/rKX8se8Z2o_7egLVVodTRHaIPUs'},
    'particulars': {'contains_fixed_charge': True,
     'contains_floating_charge': True,
     'contains_negative_pledge': True,
     'description': '.',
     'floating_charge_covers_all': True,
     'type': 'brief-description'},
    'persons_entitled': [{'name': 'Timothy Mark Jones'},
     {'name': 'Robert Peter Philip Evans'},
     {'name': 'Julie Ann Evans'}],
    'status': 'outstanding',
    'transactions': [{'delivered_on': '2026-04-13',
      'filing_type': 'create-charge-with-deed',
      'links': {'filing': '/company/17016019/filing-history/

### Charges endpoint — No-data behaviour

**Endpoint:** `/company/{n}/charges`

- **With charges:** Returns HTTP `200`, with charge records in `items` and `total_count` greater than zero.
- **Without charges:** Returns HTTP `200`, with `items: []` and `total_count: 0`.
- **Conclusion:** An empty list is a valid response, not an API error.

**MCP implementation:** Check `total_count` or `items` to determine whether charges exist. Do not rely on the HTTP status code alone.

In [21]:
RAW[TEST_SET["no_charges"]]["charges"]

{'status': 200,
 'body': {'etag': 'c7889dcf670eca9b0a6b165be8eba3ffecf26668',
  'items': [],
  'part_satisfied_count': 0,
  'satisfied_count': 0,
  'total_count': 0,
  'unfiltered_count': 0}}

In [22]:
RAW[TEST_SET["insolvency"]]["insolvency"]

{'status': 200,
 'body': {'cases': [{'dates': [{'date': '2024-09-11', 'type': 'wound-up-on'},
     {'date': '2024-09-09', 'type': 'declaration-solvent-on'}],
    'number': '1',
    'practitioners': [{'address': {'address_line_1': 'Suite 2 The Brentano Suite Solar House',
       'address_line_2': '915 High Road',
       'locality': 'London',
       'postal_code': 'N12 8QJ'},
      'appointed_on': '2024-09-11',
      'name': 'Myles Jacobson',
      'role': 'practitioner'}],
    'type': 'members-voluntary-liquidation'}],
  'etag': '62044f036e6760766914b7c889182194c5ccb71f',
  'status': ['liquidation']}}

### Insolvency endpoint — No-data behaviour

**Endpoint:** `/company/{n}/insolvency`

- **Company with insolvency:** Returns HTTP `200` with a `cases` list and insolvency status information.
- **Company without insolvency:** Returns HTTP `404` with `"error": "Not Found"`.
- **Conclusion:** For this endpoint, HTTP `404` represents the absence of insolvency data and should be treated as a valid "no data" result rather than an API error.

**MCP implementation:** Handle `404` as "no insolvency data" for this endpoint, while treating other unexpected errors separately.

In [24]:
RAW[TEST_SET["no_charges"]]["insolvency"]

{'status': 404,
 'body': {'timestamp': '2026-10-03T18:07:12.447Z',
  'status': 404,
  'error': 'Not Found',
  'path': '/company/17491796/insolvency'}}

Simply:
- Charges: No charges → 200 with an empty items list.
- Insolvency: No insolvency data → 404.

So the important lesson is:
- Different API endpoints represent "no data" differently. We need to understand each endpoint's behaviour so the MCP server doesn't mistakenly treat valid "no data" responses as errors.

## 6. Can each question Q1–Q10 be answered?


This code is now taking the raw API responses I collected earlier and turning them into answers for Q1–Q7.

In simple terms:
- Before, we collected the data. Now we're extracting the useful information from that data.

```
1. Find test companies
        ↓
2. Call API endpoints
        ↓
3. Save RAW responses
        ↓
4. Discover how "no data" behaves
        ↓
5. Extract answers Q1–Q7
        ↓
6. Put answers into a DataFrame
```

In [ ]:
def body(num, name):
    r = RAW[num][name]
    return r["body"] if r["status"] == 200 else None


def answer_q1_to_q7(num: str) -> dict:
    p = body(num, "profile") or {}
    officers = (body(num, "officers") or {}).get("items", [])
    psc = (body(num, "psc") or {}).get("items", [])
    charges = (body(num, "charges") or {}).get("items", [])
    insolvency = (body(num, "insolvency") or {}).get("cases", [])
    active_officers = [o for o in officers if not o.get("resigned_on")]
    
    return {
        "Q1_status": p.get("company_status"), # company's current status
        "Q2_created": p.get("date_of_creation"), # date the company was created
        "Q3_active_directors": [ # names of active directors
            o.get("name")
            for o in active_officers
            if o.get("officer_role") == "director"
        ],
        "Q4_accounts_overdue": p.get("accounts", {}).get("overdue"), # whether the company has overdue accounts
        "Q4_conf_stmt_overdue": p.get("confirmation_statement", {}).get("overdue"), # whether the company has overdue confirmation statement
        "Q5_psc": [(x.get("name"), x.get("kind")) for x in psc], # names and kinds of PSCs
        "Q6_charges_total": len(charges), # total number of charges
        "Q6_charges_outstanding": sum( # total number of outstanding charges
            1 for c in charges if c.get("status") == "outstanding" 
        ),
        "Q6_profile_flag_has_charges": p.get("has_charges"), # whether the company has any charges at all
        "Q7_insolvency_cases": len(insolvency), # total number of insolvency cases
        "Q7_profile_flag_insolvency": p.get("has_insolvency_history"), # whether the company has any insolvency history
    }


answers = pd.DataFrame({cat: answer_q1_to_q7(num) for cat, num in TEST_SET.items()}).T
answers.head(3)

,Q1_status,Q2_created,Q3_active_directors,Q4_accounts_overdue,Q4_conf_stmt_overdue,Q5_psc,Q6_charges_total,Q6_charges_outstanding,Q6_profile_flag_has_charges,Q7_insolvency_cases,Q7_profile_flag_insolvency
large_plc,active,1947-11-27,"[BETHELL, Melissa, BODSON, Bertrand Jean Francois, FAIRBAIRN, Carolyn Julie,...",False,False,[],9,2,False,0,False
new_ltd,active,2026-09-30,"[SAJJALA, Govardhan Reddy]",False,False,"[(Mr Govardhan Reddy Sajjala, individual-person-with-significant-control)]",0,0,False,0,False
dissolved,dissolved,2019-07-22,"[ARNSBY, David Michael]",None,None,"[(Mr David Michael Arnsby, individual-person-with-significant-control)]",0,0,False,0,False


**Check by hand:** open 2–3 of these companies on the Companies House website and confirm the answers match. This is your first taste of building ground truth (Phase 3).

**Watch for:** do the profile flags (`has_charges`, `has_insolvency_history`) agree with the detailed endpoints? If not, which should the agent trust?


In simple terms

`API data → your code → answers → manually check against Companies House website.`

This is called ground truth: we're verifying that what your agent says matches the authoritative source before trusting the agent's answers.

In [28]:
answers

,Q1_status,Q2_created,Q3_active_directors,Q4_accounts_overdue,Q4_conf_stmt_overdue,Q5_psc,Q6_charges_total,Q6_charges_outstanding,Q6_profile_flag_has_charges,Q7_insolvency_cases,Q7_profile_flag_insolvency
large_plc,active,1947-11-27,"[BETHELL, Melissa, BODSON, Bertrand Jean Francois, FAIRBAIRN, Carolyn Julie,...",False,False,[],9,2,False,0,False
new_ltd,active,2026-09-30,"[SAJJALA, Govardhan Reddy]",False,False,"[(Mr Govardhan Reddy Sajjala, individual-person-with-significant-control)]",0,0,False,0,False
dissolved,dissolved,2019-07-22,"[ARNSBY, David Michael]",None,None,"[(Mr David Michael Arnsby, individual-person-with-significant-control)]",0,0,False,0,False
insolvency,liquidation,1979-12-24,"[PADMORE, Michael William]",True,True,"[(Mr Michael William Padmore, individual-person-with-significant-control)]",0,0,False,1,True
no_charges,active,2026-09-30,"[SAJJALA, Govardhan Reddy]",False,False,"[(Mr Govardhan Reddy Sajjala, individual-person-with-significant-control)]",0,0,False,0,False
overdue_accounts,active,2019-06-18,"[OSMAN, Malcolm]",True,True,"[(Malcolm Osman, individual-person-with-significant-control)]",0,0,False,0,False
has_charges,active,2026-02-06,"[GILL, Andrew Edward, JONES, Christopher Mark]",False,False,"[(Mr Andrew Edward Gill, individual-person-with-significant-control), (Mr Ch...",1,1,True,0,False
corporate_psc,active,2000-04-20,"[BOHAN, Valentine Anthony, YI, Kenneth Hohee]",False,False,"[(Alphabet, Inc., corporate-entity-person-with-significant-control)]",1,1,False,0,False


In [27]:
TEST_SET

{'large_plc': '00445790',
 'new_ltd': '17491796',
 'dissolved': '12115175',
 'insolvency': '01468547',
 'no_charges': '17491796',
 'overdue_accounts': '12056899',
 'has_charges': '17016019',
 'corporate_psc': '03977902'}

**Can conclude:**
- API responses are being interpreted correctly.
- Q1–Q7 extraction appears to be working.
- The profile flags and detailed endpoint data agree for the companies I checked.
- I now have some ground truth to validate the MCP server against.

### Q9 — Recent changes (last 12 months)

Director changes come from officer `appointed_on` / `resigned_on`. Address changes come from filing history — find which filing `type` codes represent a registered office change.


`Now from:` "What is this company?"

`Move to:` "What changed for each company in the last 12 months?"

In [29]:
from datetime import date, timedelta

CUTOFF = (date.today() - timedelta(days=365)).isoformat()


def recent_changes(num: str) -> dict:
    officers = (body(num, "officers") or {}).get("items", [])
    filings = (body(num, "filing_history") or {}).get("items", [])
    appointed = [o["name"] for o in officers if o.get("appointed_on", "") >= CUTOFF]
    resigned = [o["name"] for o in officers if o.get("resigned_on", "") >= CUTOFF]
    recent_filings = [
        (f.get("date"), f.get("type"), f.get("category"), f.get("description"))
        for f in filings
        if f.get("date", "") >= CUTOFF
    ]
    return {
        "appointed_12m": appointed,
        "resigned_12m": resigned,
        "recent_filings": recent_filings,
    }


for cat, num in TEST_SET.items():
    print(f"\n== {cat} ({num})")
    rc = recent_changes(num)
    print("Appointed:", rc["appointed_12m"], "| Resigned:", rc["resigned_12m"])
    for f in rc["recent_filings"][:8]:
        print("  ", f)


== large_plc (00445790)
Appointed: [] | Resigned: []
   ('2026-09-16', 'SH03', 'capital', 'capital-return-purchase-own-shares')
   ('2026-08-26', 'SH06', 'capital', 'capital-cancellation-shares')
   ('2026-08-26', 'SH06', 'capital', 'capital-cancellation-shares')
   ('2026-08-26', 'SH06', 'capital', 'capital-cancellation-shares')
   ('2026-08-26', 'SH06', 'capital', 'capital-cancellation-shares')
   ('2026-08-06', 'SH03', 'capital', 'capital-return-purchase-own-shares')
   ('2026-08-06', 'SH03', 'capital', 'capital-return-purchase-own-shares')
   ('2026-08-06', 'SH03', 'capital', 'capital-return-purchase-own-shares')

== new_ltd (17491796)
Appointed: ['SAJJALA, Govardhan Reddy'] | Resigned: []
   ('2026-09-30', 'NEWINC', 'incorporation', 'incorporation-company')

== dissolved (12115175)
Appointed: [] | Resigned: []
   ('2026-09-29', 'GAZ2(A)', 'gazette', 'gazette-dissolved-voluntary')
   ('2026-07-17', 'AA', 'accounts', 'accounts-with-accounts-type-micro-entity')
   ('2026-07-17', 'AA

We use the below code to **find out which filing codes Companies House uses for different types of filings**, especially **registered-office address changes**.

In simple terms:

> It looks at the filing history of all our test companies, collects the different filing types/descriptions, counts how often they appear, and shows the most common ones in a table.

The reason is that we need to discover the **exact code that represents an address change**.

For example, we might discover:

```text
type    category    description
AD01    address     change-registered-office-address-...
```

Then we can tell the MCP agent:

> "When you see this filing code, interpret it as a registered-office address change."

So this code is basically **helping us discover the data codes we need for Q9**.

In [33]:
# Which filing types/descriptions appear across the test set? Find the one(s) for address changes.
all_types = Counter()
for num in TEST_SET.values():
    for f in (body(num, "filing_history") or {}).get("items", []):
        all_types[(f.get("type"), f.get("category"), f.get("description"))] += 1
        
pd.DataFrame(
    [(*k, v) for k, v in all_types.most_common(10)], # most_common(40)
    columns=["type", "category", "description", "count"],
)

,type,category,description,count
0,CS01,confirmation-statement,confirmation-statement-with-no-updates,21
1,AA,accounts,accounts-with-accounts-type-total-exemption-full,13
2,SH03,capital,capital-return-purchase-own-shares,12
3,SH06,capital,capital-cancellation-shares,10
4,AA,accounts,accounts-with-accounts-type-full,8
5,AA,accounts,accounts-with-accounts-type-micro-entity,7
6,CS01,confirmation-statement,confirmation-statement-with-updates,6
7,RESOLUTIONS,resolution,resolution,5
8,NEWINC,incorporation,incorporation-company,5
9,AD01,address,change-registered-office-address-company-with-date-old-address-new-address,5


Perfect. This gives us the exact information we were looking for.

The important row is:

```text
AD01 | address | change-registered-office-address-company-with-date-old-address-new-address
```

So we've discovered that **`AD01`** represents a registered-office address change in our test data.

### What to record

> **Q9 — Registered office address change:** filing type `AD01`, category `address`, with description `change-registered-office-address-company-with-date-old-address-new-address`.

This is useful because the MCP agent can now recognize this filing code and translate it into something understandable like:

**"Registered office address changed."**

We have therefore completed the **discovery part of Q9**.

**Record:** the filing `type`/`description` codes that mean "registered office address changed". Note that `description` is a **code** (e.g. `change-registered-office-address-...`), not human text — the agent will need a mapping.


## 7. Q8 — Director network (the expensive question)

For each director, get their other appointments via the `links.officer.appointments` path in the officers response, then check the status of each linked company.

This is where the API call count can explode. **Measure it.**


Simply, this step is asking:

> **For each director of our test companies, find the other companies they are connected to, then check whether those companies are active, dissolved, etc.**

For example:

```text
Company A
   ↓
Director: John Smith
   ↓
John Smith's other appointments
   ↓
Company B, Company C, Company D
   ↓
Check status of B, C, D
```

### Why is it called the "expensive question"?

Because the number of API requests can grow very quickly.

Imagine:

```text
10 test companies
   ↓
5 directors each
   ↓
50 directors
   ↓
10 appointments each
   ↓
500 other companies
   ↓
500 more API calls to check their status
```

So instead of just making a few API calls per company, you could suddenly make **hundreds or thousands**.

That's why the instruction says **"Measure it."**

We want to record something like:

```text
Test companies:       8
Directors checked:    35
Appointments found:   210
Additional API calls: 210
Total API calls:      ...
```

The purpose is to understand the **cost of answering Q8** and make sure the eventual MCP server doesn't accidentally make an excessive number of API requests.

**The key purpose is Q8 + cost measurement:**

> How many other companies are connected to the directors, and how many API calls does it take to discover that network?

This helps you understand whether Q8 could become expensive in API/rate-limit terms.

In [34]:
def director_network(num: str, max_directors: int = 5) -> tuple[pd.DataFrame, int]:
    calls_before = len(CALL_LOG)
    officers = (body(num, "officers") or {}).get("items", [])
    directors = [
        o
        for o in officers
        if o.get("officer_role") == "director" and not o.get("resigned_on")
    ][:max_directors]
    rows = []
    for d in directors:
        path = d.get("links", {}).get("officer", {}).get("appointments")
        if not path:
            continue
        status, appts = ch_get(path, {"items_per_page": 50}, label="appointments")
        if status != 200 or not appts:
            continue
        for a in appts.get("items", []):
            to = a.get("appointed_to", {})
            rows.append(
                {
                    "director": d.get("name"),
                    "linked_company": to.get("company_name"),
                    "linked_number": to.get("company_number"),
                    "linked_status": to.get("company_status"),
                    "role": a.get("officer_role"),
                    "appointed_on": a.get("appointed_on"),
                    "resigned_on": a.get("resigned_on"),
                }
            )
    return pd.DataFrame(rows), len(CALL_LOG) - calls_before


net_rows = []
for cat, num in TEST_SET.items():
    df, calls = director_network(num)
    failed = (
        df[
            df["linked_status"].isin(
                ["dissolved", "liquidation", "insolvency-proceedings", "administration"]
            )
        ]
        if not df.empty
        else df
    )
    net_rows.append(
        {
            "category": cat,
            "directors_checked": df["director"].nunique() if not df.empty else 0,
            "linked_companies": len(df),
            "failed_linked": len(failed),
            "api_calls": calls,
        }
    )
pd.DataFrame(net_rows)

,category,directors_checked,linked_companies,failed_linked,api_calls
0,large_plc,5,88,26,5
1,new_ltd,1,1,0,1
2,dissolved,1,1,1,1
3,insolvency,1,1,1,1
4,no_charges,1,1,0,1
5,overdue_accounts,1,1,0,1
6,has_charges,2,4,0,2
7,corporate_psc,2,20,9,2


This table confirms the main Q8 findings:

- **large_plc:** 5 directors → 88 linked companies → **5 API calls**.
- **corporate_psc:** 2 directors → 20 linked companies → **2 API calls**.
- The other categories required only **1 call per director**.
- Therefore, the appointments response **already provides the linked company's status**; you don't need an additional call for each linked company.
- `failed_linked` shows how many linked companies have a status such as dissolved, liquidation, administration, or insolvency proceedings.

### Key conclusion to record

> **Q8 requires approximately 1 appointments API call per active director, because the appointments response already includes `company_status` for each linked company.**

Also record the **officer-ID/name matching limitation**: the same real person can potentially appear under different officer IDs or slightly different identity details.

**Key finding to record:** Does the appointments response already include `company_status` for each linked company? If **yes**, Q8 needs only **1 call per director**. If **no**, it needs 1 extra call per linked company — which could be dozens of calls for a director with many companies.

**What I mean is:**

1. Check whether the API already gives the company status
When I ask for a director's appointments, the response might already say:
```
Company A → active
Company B → dissolved
Company C → active
```
- If it does, great. I only need:
- 1 director → 1 API call

We don't need to separately call the API for Company A, B, C just to find their statuses.

Our results show that it does include company_status, so Q8 is relatively efficient.

---

**Also record:** name matching problems. The same real person can appear under **different officer IDs** (e.g. different spelling or address). This is a known limitation to document honestly.

**What I mean is:**

Be careful identifying the same person

The same real person might appear in Companies House data as different officer records:
```
Officer ID 123 → "John Smith", London
Officer ID 456 → "Jon Smith", Manchester
```

They might actually be the same person, but the API doesn't necessarily tell you that.

So we shouldn't automatically combine them based only on name.

**In one sentence**
>Q8 is efficient because the appointments response already tells us the status of linked companies, but identifying whether different officer records represent the same real person is a limitation we must document.

## 8. Call budget, latency and rate limit


Simply, this code is measuring **how expensive your API testing has been**.

It answers three questions:

1. **How many API calls did we make?**
   ```text
   Total calls this session
   ```

2. **What status codes did each endpoint return?**
   ```text
   profile       200
   charges       200
   insolvency    200 / 404
   ```
   This helps identify normal responses vs no-data responses.

3. **How long does each endpoint take?**
   It calculates:
   - average time
   - median (`50%`)
   - `95%` of requests are faster than this (`95%`)
   - maximum time

For example:

```text
endpoint       calls   mean   50%   95%   max
profile          8     120    110   180   250
officers         8     150    140   220   300
```

### Why are we doing this?

Because the MCP server will eventually need to operate within **API call limits and reasonable response times**.

So you're measuring:

> **How many requests does this workflow require, which responses occur, and how long do the requests take?**

This gives you evidence for deciding where you need **limits, caching, or optimization**.

In [ ]:
log = pd.DataFrame(CALL_LOG)
print("Total calls this session:", len(log)) # How many API calls did we make?
print("\nStatus codes by endpoint:") # What status codes did we get for each endpoint?
display(log.groupby(["label", "status"]).size().unstack(fill_value=0))  # This helps identify normal responses vs no-data responses.

print("\nLatency by endpoint (ms):") # How long does each endpoint take?
display(
    log.groupby("label")["ms"]
    .describe(percentiles=[0.5, 0.95])[["count", "mean", "50%", "95%", "max"]]
    .round(0)
)

Total calls this session: 109

Status codes by endpoint:


status,200,404
label,,
appointments,14,0
charges,8,0
filing_history,8,0
insolvency,1,7
officers,8,0
profile,49,0
psc,8,0
search,6,0



Latency by endpoint (ms):


,count,mean,50%,95%,max
label,,,,,
appointments,14.0,63.0,48.0,130.0,219.0
charges,8.0,42.0,41.0,50.0,53.0
filing_history,8.0,68.0,39.0,171.0,215.0
insolvency,8.0,31.0,29.0,36.0,38.0
officers,8.0,46.0,40.0,60.0,62.0
profile,49.0,40.0,28.0,115.0,219.0
psc,8.0,59.0,58.0,78.0,82.0
search,6.0,193.0,187.0,272.0,279.0


This code is estimating **how expensive one complete Q1–Q10 report would be** in terms of API calls and time.

### In simple terms

It calculates:

- **How many API calls one report normally needs**
- **The maximum number of calls**
- **How many reports could theoretically fit within a 600-call / 5-minute limit**
- **Typical API response time**
- **Estimated sequential time for one report**

The important part is:

```text
6 base endpoints
+
Q8 appointment calls
=
total calls for one report
```

For example, if Q8 needs 5 calls:

```text
6 + 5 = 11 API calls
```

So the code is basically asking:

> **"If a user asks our MCP server for a complete company report, how many API requests will we make, and how long might it take?"**

This is important for designing the MCP server because you need to know whether **rate limits and response time** could become a problem.

One caveat: the calculation uses the **median call count** and a **single overall median latency**, so it's a rough estimate rather than a precise prediction for every report.

In [38]:
# Estimate calls and time for ONE full report (Q1–Q10)
base_calls = len(ENDPOINTS)  # profile, officers, filings, psc, charges, insolvency
q8_calls = pd.DataFrame(net_rows)["api_calls"]
per_report = base_calls + q8_calls
p95_ms = log["ms"].quantile(0.95)

summary = {
    "calls_per_report_median": float(per_report.median()),
    "calls_per_report_max": float(per_report.max()),
    "reports_per_5min_at_600_limit": (
        int(600 // per_report.median()) if per_report.median() else None
    ),
    "api_p95_ms_per_call": round(p95_ms),
    "sequential_api_time_s_median_report": round(
        per_report.median() * log["ms"].median() / 1000, 1
    ),
}
summary

{'calls_per_report_median': 7.0,
 'calls_per_report_max': 11.0,
 'reports_per_5min_at_600_limit': 85,
 'api_p95_ms_per_call': 189,
 'sequential_api_time_s_median_report': np.float64(0.3)}

Your output means:

- **Median calls per report: 7** → a typical full report needs about **7 API calls**.
- **Maximum: 11** → the most expensive test case needed **11 calls**.
- **85 reports / 5 minutes** → based on the 600-call limit, you could theoretically handle about **85 typical reports** in 5 minutes.
- **95th-percentile API call: 189 ms** → 95% of individual calls took **189 ms or less** in your test.
- **Median report time: 0.3 seconds** → a typical report's sequential API calls took roughly **0.3 seconds** based on this estimate.

### Main conclusion

> **The API itself is reasonably fast, and a typical report only needs about 7 calls. Q8 is what can increase the cost, with the maximum reaching 11 calls in your test.**

So this gives you evidence that the MCP server can be designed around a **rough budget of 7–11 API calls per full report**, while being careful with Q8.

**Questions to answer from this:**

- Can a full report stay under the **30-second** target (metric M7) once LLM time is added? If sequential calls are slow, which calls can run **in parallel**?
- How many reports can run per 5 minutes? This decides how fast the **eval suite** can run — and is why caching and recorded fixtures are essential.


Simply:

### 1. Can it stay under 30 seconds?
**Yes, probably.**

Your API calls are quite fast:
- Typical report = about **7 API calls**
- Worst tested = **11 calls**
- API time was only about **0.3 seconds** sequentially.

The calls for **profile, officers, filings, PSC, charges, and insolvency can mostly run in parallel**, making the report even faster.

The main unknown is **how long the LLM takes** to generate the final report. So you need to test the full process, not just the API.

### 2. How many reports in 5 minutes?
With the **600-call limit**:

- 7 calls/report → about **85 reports / 5 minutes**
- 11 calls/report → about **54 reports / 5 minutes**

But these are theoretical limits.

### 3. Why caching/fixtures?
Because running every eval against the real API would:
- use up the API rate limit
- make the eval slower
- give inconsistent results

So **record the API responses once, then reuse them for evals**.

**Simple conclusion:**  
> Parallel API calls + caching should make the system fast enough for the 30-second target, while recorded fixtures let you run lots of evals without repeatedly calling Companies House.

## 9. Findings → copy into `docs/data-notes.md`


In [46]:
findings = {
    "auth_works": True,
    "rate_limit_headers": (
        log[["ratelimit_limit", "ratelimit_window"]].dropna().iloc[-1].to_dict()
        if not log[["ratelimit_limit"]].dropna().empty
        else "none seen"
    ),
    "no_data_status_codes": log[log["status"] != 200]
    .groupby("label")["status"]
    .unique()
    .apply(list)
    .to_dict(),
    "test_set": TEST_SET,
    **summary,
}
print(json.dumps(findings, indent=2, default=str))
# save_raw("_spike_findings", findings) # is saving the file using your existing save_raw() function, which apparently writes to a JSON file in the raw data directory.


with open(DOCS_DIR / "05-data-notes-summary.md", "w", encoding="utf-8") as f:
    f.write("# Data Notes\n\n")
    f.write("## Spike Findings\n\n")
    f.write("```json\n")
    f.write(json.dumps(findings, indent=2, default=str))
    f.write("\n```\n")

print("Saved to docs/05-data-notes-summary.md")

{
  "auth_works": true,
  "rate_limit_headers": {
    "ratelimit_limit": "600",
    "ratelimit_window": "5m"
  },
  "no_data_status_codes": {
    "insolvency": [
      "404"
    ]
  },
  "test_set": {
    "large_plc": "00445790",
    "new_ltd": "17491796",
    "dissolved": "12115175",
    "insolvency": "01468547",
    "no_charges": "17491796",
    "overdue_accounts": "12056899",
    "has_charges": "17016019",
    "corporate_psc": "03977902"
  },
  "calls_per_report_median": 7.0,
  "calls_per_report_max": 11.0,
  "reports_per_5min_at_600_limit": 85,
  "api_p95_ms_per_call": 189,
  "sequential_api_time_s_median_report": 0.3
}
Saved to docs/05-data-notes-summary.md


In [45]:
data_notes = f"""# Data Notes — Companies House API

| Field | Value |
|---|---|
| Version | 0.1 (Spike results) |
| Owner | Fraidoon Omarzai |
| Status | ✅ Spike completed |

---

## 1. Access

| Item | Value |
|---|---|
| Base URL | `https://api.company-information.service.gov.uk` |
| Auth | HTTP Basic — API key as username, empty password |
| Key type | REST API key, live environment |
| Rate limit | 600 requests / 5 minutes per key |
| Rate-limit response | HTTP 429 |
| Cost | Free |
| Licence | Open Government Licence v3.0 — attribution required |

## 2. Endpoints

| Endpoint | Purpose | "No data" behaviour | Paginated? | Notes |
|---|---|---|---|---|
| `GET /search/companies?q=` | Find company by name | 200 + empty `items` | Yes | |
| `GET /company/{{n}}` | Profile | 404 if company does not exist | No | |
| `GET /company/{{n}}/officers` | Directors, secretaries | 200 + `items` | Yes | |
| `GET /company/{{n}}/filing-history` | Filings | 200 + `items` | Yes | `description` contains filing codes |
| `GET /company/{{n}}/persons-with-significant-control` | Owners / controllers | 200 + `items` | Yes | |
| `GET /company/{{n}}/charges` | Secured debts | 200 + empty `items` when no charges | Yes | |
| `GET /company/{{n}}/insolvency` | Insolvency cases | 404 when no insolvency record | No | |
| `GET /officers/{{id}}/appointments` | A person's appointments | 200 + `items` | Yes | Path comes from `links.officer.appointments` |

## 3. Question → data mapping

Status: ✅ answerable · 🟡 partial · ❌ not answerable

| ID | Question | Endpoint(s) | Field(s) | Status | Notes |
|---|---|---|---|---|---|
| Q1 | Active / dissolved / liquidation? | profile | `company_status`, `date_of_cessation` | ✅ | |
| Q2 | How old? | profile | `date_of_creation` | ✅ | |
| Q3 | Current directors? | officers | `name`, `officer_role`, `appointed_on`, `resigned_on` | ✅ | Filter out resigned |
| Q4 | Accounts / confirmation statement overdue? | profile | `accounts.overdue`, `confirmation_statement.overdue` | ✅ | |
| Q5 | Who controls it? | psc | `name`, `kind`, `natures_of_control` | ✅ | Corporate PSCs identified |
| Q6 | Charges? | charges + profile | `status`, `classification`, `created_on`, `has_charges` | ✅ | 200 + empty items means no charges |
| Q7 | Insolvency history? | insolvency + profile | `cases`, `has_insolvency_history` | ✅ | 404 can mean no insolvency record |
| Q8 | Directors linked to failed companies? | officers → appointments | `appointed_to.company_status` | ✅ | Appointment calls per active director |
| Q9 | Recent director/address changes? | officers, filing-history | dates; filing codes | ✅ | `AD01` identifies registered-office address changes |
| Q10 | Red flags summary | Q1–Q9 | — | 🟡 | Synthesis layer |

## 4. Test set used in the spike

| Category | Company number | Notes |
|---|---|---|
| Large plc | {TEST_SET.get("large_plc", "")} | Lots of data |
| New Ltd (< 1 year) | {TEST_SET.get("new_ltd", "")} | Little data |
| Dissolved | {TEST_SET.get("dissolved", "")} | Status handling |
| Insolvency / liquidation | {TEST_SET.get("insolvency", "")} | Insolvency endpoint |
| Overdue accounts | {TEST_SET.get("overdue_accounts", "")} | Overdue flag |
| Has charges | {TEST_SET.get("has_charges", "")} | Charges present |
| No charges | {TEST_SET.get("no_charges", "")} | Empty charges response |
| Corporate PSC | {TEST_SET.get("corporate_psc", "")} | Corporate PSC case |

## 5. Measurements

| Measure | Value |
|---|---|
| API calls per full report (median / max) | {summary["calls_per_report_median"]} / {summary["calls_per_report_max"]} |
| Q8 calls per report (median / max) | {float(q8_calls.median())} / {int(q8_calls.max())} |
| Reports per 5 minutes within rate limit | {summary["reports_per_5min_at_600_limit"]} theoretical |
| API latency per call (median / p95, ms) | {log["ms"].median():.0f} / {summary["api_p95_ms_per_call"]} |
| Estimated sequential API time per report (s) | {summary["sequential_api_time_s_median_report"]} |
| Calls that could run in parallel | Profile, officers, filing history, PSC, charges, insolvency |

## 6. Quirks and gotchas

- Charges with no data return **HTTP 200 with an empty `items` list**.
- Insolvency with no data returns **HTTP 404**.
- Different endpoints therefore use different "no data" behaviour.
- Q8 requires the officers response before appointment lookups can begin.
- Appointment responses already contain the linked company's `company_status`.
- The same person may appear under multiple officer IDs, so identity matching needs care.
- Filing history uses machine-readable codes such as `AD01`; these need mapping to plain English.
- `AD01` was observed for registered-office address changes.
- Q8 can create many linked-company results, but does not require one extra API call per linked company.

## 7. Decisions triggered by the spike

| Decision | Reason | Logged in decision log? |
|---|---|---|
| Limit Q8 to a manageable number of directors | Controls API call budget | |
| Treat 200 + empty charges list as "no charges" | Observed behaviour | |
| Treat insolvency 404 as "no insolvency record" | Observed behaviour | |
| Run independent API calls in parallel | Reduces report latency | |
| Use cached/recorded fixtures for evals | Avoid repeated API usage | |

## 8. Assumptions updated

| ID | Assumption | Result |
|---|---|---|
| A4 | Data can answer Q1–Q10 | 🟡 Mostly supported; Q10 is synthesis |
| A5 | Report possible in < 30 s within rate limits | 🟢 API portion has substantial headroom; LLM time still needs measurement |

---

## 9. Spike Summary

### Authentication
- API authentication worked successfully.
- Rate-limit headers reported **600 requests / 5 minutes**.

### API behaviour
- Most tested endpoints returned HTTP 200.
- Charges with no records returned **200 + empty `items`**.
- Insolvency with no record returned **404**.

### Performance
- Typical report required approximately **7 API calls**.
- Maximum observed was **11 API calls**.
- Median API latency was approximately **{log["ms"].median():.0f} ms**.
- 95th-percentile API latency was **{summary["api_p95_ms_per_call"]} ms**.
- Estimated sequential API time was approximately **{summary["sequential_api_time_s_median_report"]} seconds**.
- Theoretical capacity at 7 calls/report and a 600-call limit is approximately **85 reports per 5 minutes**.

### Parallelisation
The main independent calls — profile, officers, filing history, PSC, charges, and insolvency — can largely run in parallel. Director appointment lookups must happen after officers are retrieved.

### Evaluation strategy
Repeated evals should use **cached responses or recorded fixtures** rather than calling the live Companies House API every time. This avoids rate-limit usage and makes eval results faster and more reproducible.

### Overall conclusion
The Companies House API portion appears fast enough to support the **30-second M7 target**. The remaining measurement needed is the **full end-to-end time including LLM generation**.

"""

Path(DOCS_DIR / "05-data-notes.md").write_text(data_notes, encoding="utf-8")

print("Saved full data notes to docs/05-data-notes.md")

Saved full data notes to docs/05-data-notes.md


### Checklist before closing the spike

- [ ] Every question Q1–Q10 marked **answerable / partial / not answerable** in `05-data-notes.md`
- [ ] "No data" behaviour recorded for every endpoint
- [ ] Q8 call cost measured and a limit decided (e.g. max directors, max appointments per director)
- [ ] Address-change filing codes identified (Q9)
- [ ] Rate limit confirmed and reports-per-5-min calculated
- [ ] Assumptions A4 and A5 in `01-problem-statement.md` updated (confirmed / rejected)
- [ ] Any scope changes logged in `docs/kickoff/decision-log.md`
